In [ ]:
import os
import sys
print(sys.executable)

In [ ]:
user = os.environ["USER"]

os.environ["MIOPEN_CUSTOM_CACHE_DIR"] = f"/tmp/miopen-cache-{user}"
os.environ["MIOPEN_USER_DB_PATH"] = f"/tmp/miopen-config-{user}"

os.makedirs(os.environ["MIOPEN_CUSTOM_CACHE_DIR"], exist_ok=True)
os.makedirs(os.environ["MIOPEN_USER_DB_PATH"], exist_ok=True)

### Install Additional Python Packages

Open a terminal in JupyterLab and run:

**1. Create and activate virtual environment:**

```bash
python -m venv my-env --system-site-packages
source my-env/bin/activate
```

**2. Install hackathon dependencies:**

```bash
pip list
pip install --upgrade pip

pip install langchain langchain-openai langchain-community \
langchain-text-splitters openai python-dotenv pydantic \
pandas pymupdf arxiv chromadb streamlit plotly pytest
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Define question
messages = [
    {
        "role": "user",
        "content": "Explain how quantum computing works in simple terms."
    }
]

# Apply chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

# Move inputs to model device
inputs = inputs.to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Extract generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

# Decode response
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

# Print result
print("\nQuestion: Explain how quantum computing works in simple terms.\n")
print("Qwen Response:\n")
print(response)